# LearnMateAI — Candidate Adapter Evaluation

Loads a LoRA adapter + its **run-record**, scores it on the held-out **test** split, compares against the **fallback** model, and writes a pass/fail row to `version_registry.csv`.

Acceptance contract: `acceptance_thresholds.yaml`.

**Status:** Logic is complete. A real GPU + live Gemini comparison has **not** been executed in-repo. Use `--dry-run-metrics` style cells below for registry plumbing tests without a model.

## 0 — Install (Colab)

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q transformers==4.44.2 peft==0.12.0 accelerate==0.33.0 \
        bitsandbytes==0.43.3 sentencepiece pyyaml pandas openai python-dotenv
print("IN_COLAB =", IN_COLAB)

## 1 — Evaluation CONFIG

In [ ]:
from pathlib import Path

EVAL_CONFIG = {
    # Point at a trained adapter directory that contains adapter weights + run_record.json
    "adapter_dir": "../02_finetuning/adapters/REPLACE_WITH_RUN_ID/adapter",
    "run_record_path": None,  # default: <adapter_dir>/run_record.json
    "test_path": "../02_finetuning/sample_data/test.jsonl",
    "thresholds_path": "acceptance_thresholds.yaml",
    "registry_path": "version_registry.csv",
    "max_new_tokens": 256,
    "temperature": 0.1,
    # If True, skip model load and inject placeholder metrics (registry/checklist plumbing only)
    "dry_run": True,
    "candidate_id": None,  # auto from run_id if None
}

print(EVAL_CONFIG)

## 2 — Load thresholds + run-record (hard gate)

In [ ]:
import json
from pathlib import Path
import yaml

with open(EVAL_CONFIG["thresholds_path"], encoding="utf-8") as f:
    THRESHOLDS = yaml.safe_load(f)

adapter_dir = Path(EVAL_CONFIG["adapter_dir"])
run_record_path = Path(EVAL_CONFIG["run_record_path"] or (adapter_dir / "run_record.json"))

if not EVAL_CONFIG["dry_run"]:
    assert run_record_path.exists(), (
        f"Missing run-record at {run_record_path}. "
        "Adapters without run-records are ineligible for evaluation."
    )
    with run_record_path.open(encoding="utf-8") as f:
        RUN_RECORD = json.load(f)
else:
    # Dry-run placeholder so registry plumbing can be demonstrated
    RUN_RECORD = {
        "run_id": "dry-run-example",
        "base_model_id": "Qwen/Qwen2.5-1.5B-Instruct",
        "dataset_version": "lm-legal-smoke-v1",
        "status": "dry_run",
    }

CANDIDATE_ID = EVAL_CONFIG["candidate_id"] or RUN_RECORD["run_id"]
print("candidate:", CANDIDATE_ID)
print("thresholds version:", THRESHOLDS["version"])
RUN_RECORD

## 3 — Load test set

In [ ]:
import json

def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

test_rows = load_jsonl(EVAL_CONFIG["test_path"])
print("test items:", len(test_rows))
assert test_rows, "Empty test set"
test_rows[0]["pair_type"], test_rows[0]["subject_area"]

## 4 — Helpers: metrics

In [ ]:
import re
import time
from collections import Counter

SECTION_CITE_RE = re.compile(r"\b(?:section|s\.|art(?:icle)?)\s*\d+[A-Za-z]?\b", re.I)

def tokenize(text: str):
    return re.findall(r"[a-z0-9]+", text.lower())

def token_f1(pred: str, gold: str) -> float:
    p, g = tokenize(pred), tokenize(gold)
    if not p and not g:
        return 1.0
    if not p or not g:
        return 0.0
    pc, gc = Counter(p), Counter(g)
    overlap = sum((pc & gc).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(p)
    recall = overlap / len(g)
    return 2 * precision * recall / (precision + recall)

def mcq_letter(text: str):
    m = re.search(r"\b([ABCD])\b", text.upper())
    return m.group(1) if m else None

def is_correct(pair_type: str, pred: str, gold: str, f1_thresh: float) -> bool:
    if pair_type == "mcq":
        return mcq_letter(pred) is not None and mcq_letter(pred) == mcq_letter(gold)
    return token_f1(pred, gold) >= f1_thresh

def hallucination_flag(pred: str, source: str) -> bool:
    """Heuristic: cited section numbers in the answer that never appear in the source."""
    src_cites = {m.group(0).lower() for m in SECTION_CITE_RE.finditer(source)}
    pred_cites = {m.group(0).lower() for m in SECTION_CITE_RE.finditer(pred)}
    unsupported = pred_cites - src_cites
    # Also flag empty answers as ungrounded
    if not pred.strip():
        return True
    return len(unsupported) > 0

def extract_user_and_gold(row):
    msgs = row["messages"]
    user = next(m["content"] for m in msgs if m["role"] == "user")
    gold = next(m["content"] for m in msgs if m["role"] == "assistant")
    source = user.split("---SOURCE EXCERPT---")[-1].strip() if "---SOURCE EXCERPT---" in user else user
    return user, gold, source

print("metric helpers ready")

## 5 — Load candidate model (skipped in dry_run)

In [ ]:
candidate_generate = None

if not EVAL_CONFIG["dry_run"]:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel

    base_id = RUN_RECORD["base_model_id"]
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    base = AutoModelForCausalLM.from_pretrained(
        base_id, quantization_config=bnb, device_map="auto", trust_remote_code=True
    )
    model = PeftModel.from_pretrained(base, str(adapter_dir))
    model.eval()

    def candidate_generate(user_text: str) -> str:
        messages = [
            {
                "role": "system",
                "content": (
                    "You are LearnMateAI, a study assistant for Sri Lankan legal education. "
                    "Answer from the provided source excerpt. If the excerpt is insufficient, say so."
                ),
            },
            {"role": "user", "content": user_text},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=EVAL_CONFIG["max_new_tokens"],
                temperature=EVAL_CONFIG["temperature"],
                do_sample=EVAL_CONFIG["temperature"] > 0,
                pad_token_id=tokenizer.pad_token_id,
            )
        gen = out[0][inputs["input_ids"].shape[-1]:]
        return tokenizer.decode(gen, skip_special_tokens=True).strip()

    print("candidate model ready")
else:
    print("dry_run=True — model load skipped")

## 6 — Fallback model client (Gemini / OpenAI-compatible)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
fb = THRESHOLDS["metrics"]["fallback_comparison"]
FALLBACK_MODEL = os.getenv(fb["fallback_env_model"], fb["default_fallback_model"])
fallback_generate = None
FALLBACK_AVAILABLE = False

if not EVAL_CONFIG["dry_run"]:
    api_key = os.getenv(fb["fallback_env_api_key"]) or os.getenv("LM_API_KEY")
    if not api_key:
        raise RuntimeError(
            "Fallback comparison is mandatory. Set GEMINI_API_KEY (or LM_API_KEY + LM_API_BASE)."
        )
    from openai import OpenAI
    # Gemini OpenAI-compatible endpoint example:
    # LM_API_BASE=https://generativelanguage.googleapis.com/v1beta/openai/
    base_url = os.getenv("LM_API_BASE") or "https://generativelanguage.googleapis.com/v1beta/openai/"
    client = OpenAI(api_key=api_key, base_url=base_url)

    def fallback_generate(user_text: str) -> str:
        resp = client.chat.completions.create(
            model=FALLBACK_MODEL,
            temperature=0.1,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a study assistant for Sri Lankan legal education. "
                        "Answer only from the provided source excerpt."
                    ),
                },
                {"role": "user", "content": user_text},
            ],
        )
        return (resp.choices[0].message.content or "").strip()

    FALLBACK_AVAILABLE = True
    print("fallback ready:", FALLBACK_MODEL)
else:
    print("dry_run=True — fallback client skipped")

## 7 — Run evaluation

In [ ]:
import statistics

f1_thresh = THRESHOLDS["metrics"]["accuracy"]["f1_pass_threshold"]

def score_predictions(preds, rows):
    correct = 0
    halluc = 0
    latencies = []
    for pred_item, row in zip(preds, rows):
        pred, latency = pred_item
        user, gold, source = extract_user_and_gold(row)
        correct += int(is_correct(row["pair_type"], pred, gold, f1_thresh))
        halluc += int(hallucination_flag(pred, source))
        latencies.append(latency)
    n = len(rows)
    accuracy = correct / n
    hall_rate = halluc / n
    groundedness = 1.0 - hall_rate
    p95 = sorted(latencies)[max(0, int(0.95 * (n - 1)))] if latencies else float("nan")
    return {
        "accuracy": accuracy,
        "groundedness": groundedness,
        "hallucination_rate": hall_rate,
        "latency_p95_ms": p95,
        "n": n,
    }

if EVAL_CONFIG["dry_run"]:
    # Illustrating registry write path only — NOT a real evaluation
    candidate_metrics = {
        "accuracy": 0.0,
        "groundedness": 0.0,
        "hallucination_rate": 1.0,
        "latency_p95_ms": 99999,
        "n": len(test_rows),
    }
    fallback_metrics = {
        "accuracy": 0.0,
        "groundedness": 0.0,
        "hallucination_rate": 1.0,
        "latency_p95_ms": 0,
        "n": len(test_rows),
    }
    FALLBACK_MODEL = "dry-run-fallback"
    print("DRY RUN metrics (intentionally failing thresholds)")
else:
    cand_preds = []
    fb_preds = []
    for row in test_rows:
        user, gold, source = extract_user_and_gold(row)
        t0 = time.perf_counter()
        pred = candidate_generate(user)
        dt = (time.perf_counter() - t0) * 1000
        cand_preds.append((pred, dt))

        t1 = time.perf_counter()
        fb_pred = fallback_generate(user)
        fb_dt = (time.perf_counter() - t1) * 1000
        fb_preds.append((fb_pred, fb_dt))

    candidate_metrics = score_predictions(cand_preds, test_rows)
    fallback_metrics = score_predictions(fb_preds, test_rows)

print("candidate:", candidate_metrics)
print("fallback :", fallback_metrics)

## 8 — Apply acceptance thresholds (fail-closed)

In [ ]:
m = THRESHOLDS["metrics"]
fail_reasons = []

if candidate_metrics["accuracy"] < m["accuracy"]["minimum"]:
    fail_reasons.append(
        f"accuracy {candidate_metrics['accuracy']:.3f} < {m['accuracy']['minimum']}"
    )
if candidate_metrics["groundedness"] < m["groundedness"]["minimum"]:
    fail_reasons.append(
        f"groundedness {candidate_metrics['groundedness']:.3f} < {m['groundedness']['minimum']}"
    )
if candidate_metrics["hallucination_rate"] > m["hallucination_rate"]["maximum"]:
    fail_reasons.append(
        f"hallucination_rate {candidate_metrics['hallucination_rate']:.3f} > {m['hallucination_rate']['maximum']}"
    )
if candidate_metrics["latency_p95_ms"] > m["latency_p95_ms"]["maximum_ms"]:
    fail_reasons.append(
        f"latency_p95_ms {candidate_metrics['latency_p95_ms']:.0f} > {m['latency_p95_ms']['maximum_ms']}"
    )

# Mandatory fallback comparison
if EVAL_CONFIG["dry_run"]:
    fail_reasons.append("dry_run=True — not a production evaluation")
elif not FALLBACK_AVAILABLE:
    fail_reasons.append("fallback comparison missing")
else:
    slack = m["fallback_comparison"]["accuracy_slack"]
    ok = (
        candidate_metrics["accuracy"] >= fallback_metrics["accuracy"]
        or (
            candidate_metrics["groundedness"] >= fallback_metrics["groundedness"]
            and candidate_metrics["accuracy"] >= fallback_metrics["accuracy"] - slack
        )
    )
    if not ok:
        fail_reasons.append(
            "failed fallback comparison "
            f"(cand_acc={candidate_metrics['accuracy']:.3f}, fb_acc={fallback_metrics['accuracy']:.3f}, "
            f"cand_ground={candidate_metrics['groundedness']:.3f}, fb_ground={fallback_metrics['groundedness']:.3f})"
        )

PASSED = len(fail_reasons) == 0
print("PASSED" if PASSED else "FAILED")
for r in fail_reasons:
    print(" -", r)

## 9 — Append version registry row

In [ ]:
import csv
from datetime import datetime, timezone
from pathlib import Path

registry_path = Path(EVAL_CONFIG["registry_path"])
fieldnames = [
    "candidate_id", "run_id", "base_model", "dataset_version", "evaluated_at_utc",
    "accuracy", "groundedness", "hallucination_rate", "latency_p95_ms",
    "fallback_model", "fallback_accuracy", "fallback_groundedness",
    "passed", "fail_reasons", "notes",
]
row = {
    "candidate_id": CANDIDATE_ID,
    "run_id": RUN_RECORD.get("run_id", CANDIDATE_ID),
    "base_model": RUN_RECORD.get("base_model_id", ""),
    "dataset_version": RUN_RECORD.get("dataset_version", ""),
    "evaluated_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "accuracy": f"{candidate_metrics['accuracy']:.4f}",
    "groundedness": f"{candidate_metrics['groundedness']:.4f}",
    "hallucination_rate": f"{candidate_metrics['hallucination_rate']:.4f}",
    "latency_p95_ms": f"{candidate_metrics['latency_p95_ms']:.1f}",
    "fallback_model": FALLBACK_MODEL,
    "fallback_accuracy": f"{fallback_metrics['accuracy']:.4f}",
    "fallback_groundedness": f"{fallback_metrics['groundedness']:.4f}",
    "passed": str(PASSED),
    "fail_reasons": " | ".join(fail_reasons),
    "notes": "dry_run" if EVAL_CONFIG["dry_run"] else "",
}

text = registry_path.read_text(encoding="utf-8") if registry_path.exists() else ""
needs_header = text.strip() == ""
# Header-only file from repo scaffold: append data row without rewriting header
with registry_path.open("a", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    if needs_header:
        writer.writeheader()
    writer.writerow(row)

print("PASSED" if PASSED else "FAILED", "| registry row appended")
print(row)
print("wrote", registry_path.resolve())



## Next

If `PASSED`, complete every step in `promotion_checklist.md` before changing the live model pointer.  
If `FAILED`, do **not** promote — fix data/training and open a new candidate (new `run_id`).